# MC-dropout — bất định epistemic từ 5 checkpoint đã có

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

**Không train gì cả.** Notebook này chỉ chạy inference: nạp `best.pt` của từng fold,
bật lại dropout, forward `K` lượt trên tập val của chính fold đó, lưu `(K, N, 7)`.

**Vì sao không dùng thẳng 5 checkpoint làm deep ensemble.** Mỗi ca ở val của fold `f`
nằm trong tập train của **cả 4 model kia** (kiểm trên `splits/`, WORKLOG S-080). Gộp
5 model rồi chấm trên 394 ca là để 4/5 thành viên chấm bài họ đã học thuộc. MC-dropout
né đúng chỗ đó: mọi thành viên đều là cùng một model của fold đó, nên đều mù với val.

**Đổi lại:** MC-dropout là xấp xỉ nghèo hơn deep ensemble thật — các thành viên chung
một cực tiểu nên đa dạng ít. Đây là phép đo rẻ để quyết có đáng đốt 4 session Kaggle
cho ensemble nhiều seed hay không, chứ không phải bản thay thế.

**Ngân sách:** ~8 phút GPU cho cả 5 fold ở `K=20`. So với 37.5h của ensemble 3 seed.

## 0. Bootstrap

Giống notebook 07. Dòng `repo commit` là bằng chứng đang chạy đúng bản code nào.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

# ---- THAM SỐ ---------------------------------------------------------------
FOLDS = [1, 2, 3, 4, 5]
N_PASSES = 20          # số lượt forward mỗi ca
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

os.environ["LLDMMRI_OUTPUT_DIR"] = "/kaggle/working/runs/mc_dropout"
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

CFG_PATH = REPO / "configs" / "baseline_3dpatch.yaml"
CFG = load_yaml(CFG_PATH)
print("dropout_prob trong config:", CFG["model"].get("dropout_prob"))

## 1. Cache E4 và checkpoint

Cần **hai** thứ mount vào: cache E4 (`lesion_tight · 112×112×32 · per_phase`) và 5 file
`best.pt`. Đổi đường dẫn bên dưới cho khớp tên dataset bạn đã upload.

In [ ]:
CACHE_CANDIDATES = [
    Path("/kaggle/input/lld-mmri-e4-per-phase"),
    Path("/kaggle/input/lld-mmri-e4"),
]
CKPT_CANDIDATES = [
    Path("/kaggle/input/lld-mmri-e4-checkpoints"),
    Path("/kaggle/input/e4-cv-results"),
]

def find_dir(candidates, marker):
    for cand in candidates:
        if not cand.exists():
            continue
        if (cand / marker).exists() or any(cand.glob(f"*/{marker}")):
            return cand
        for sub in cand.iterdir():
            if sub.is_dir() and ((sub / marker).exists() or any(sub.glob(f"*/{marker}"))):
                return sub
    return None

CACHE_DIR = find_dir(CACHE_CANDIDATES, "cache_meta.json")
CKPT_ROOT = find_dir(CKPT_CANDIDATES, "best.pt")
assert CACHE_DIR is not None, f"không thấy cache E4 trong {CACHE_CANDIDATES}"
assert CKPT_ROOT is not None, f"không thấy best.pt trong {CKPT_CANDIDATES}"
os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)
print("cache:", CACHE_DIR)
print("checkpoint:", CKPT_ROOT)

## Cổng A ⚠️ — cache có đúng là E4 không

Chạy MC-dropout trên cache của E1 hay E3 sẽ **không báo lỗi gì cả**, chỉ lặng lẽ cho ra
số sai. Giống cổng ở notebook 07.

In [ ]:
import json

meta = json.loads((Path(os.environ["LLDMMRI_CACHE_DIR"]) / "cache_meta.json").read_text("utf-8"))
EXPECTED = {
    "align_phases": "per_phase",
    "target_size": [112, 112, 32],
    "crop_mode": "lesion_tight",
}
for key, want in EXPECTED.items():
    got = meta.get(key)
    assert got == want, f"cache SAI: {key} = {got!r}, cần {want!r}. Đây không phải cache E4."
assert meta["lesion_tight"]["source"] == "mask"
print("cache_meta khớp E4 ✓  ·", meta.get("git_commit"))

## Cổng B ⚠️⚠️ — model có dropout thật không

Đây là cổng quan trọng nhất của notebook. Nếu model không có lớp Dropout nào thì `K`
lượt forward cho ra `K` kết quả **giống hệt nhau**, epistemic bằng 0 khắp nơi, và bảng
kết quả vẫn in ra bình thường — một chế độ hỏng hoàn toàn im lặng.

In [ ]:
import torch

from src.eval.mc_dropout import count_dropout_modules, enable_dropout
from src.models import build_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

probe = build_model(CFG["model"])
n_drop = count_dropout_modules(probe)
print(f"số lớp Dropout trong model: {n_drop}")
assert n_drop > 0, (
    "model KHÔNG có lớp Dropout nào — MC-dropout sẽ không làm gì cả. "
    "Kiểm model.dropout_prob trong config."
)

# Và kiểm BatchNorm vẫn ở eval sau khi bật dropout (bẫy chính, xem docstring module).
enable_dropout(probe)
bn = [m for m in probe.modules() if isinstance(m, torch.nn.modules.batchnorm._BatchNorm)]
assert bn and not any(m.training for m in bn), "BatchNorm phải ở eval sau enable_dropout"
print(f"{len(bn)} lớp BatchNorm, tất cả ở eval ✓")
del probe

## 2. Chạy MC-dropout từng fold

`build_loaders` dựng val loader với `shuffle=False`, đúng thứ cần: các pass phải xếp ca
cùng thứ tự. `mc_dropout_predict` tự kiểm điều đó và nổ nếu lệch.

In [ ]:
import time

import numpy as np

from src.eval.mc_dropout import mc_dropout_predict, save_member_probs
from src.eval.selective import uncertainty_decomposition
from src.eval.metrics import macro_f1
from src.train.run import build_loaders

OUT_ROOT = Path(os.environ["LLDMMRI_OUTPUT_DIR"])

def find_ckpt(fold: int) -> Path:
    hits = sorted(CKPT_ROOT.glob(f"fold*{fold}/best.pt")) or sorted(
        CKPT_ROOT.glob(f"**/fold_{fold}/best.pt")
    )
    assert hits, f"không thấy best.pt của fold {fold} dưới {CKPT_ROOT}"
    return hits[0]

for fold in FOLDS:
    t0 = time.time()
    _, val_loader, _ = build_loaders(CFG, fold)
    model = build_model(CFG["model"]).to(DEVICE)

    state = torch.load(find_ckpt(fold), map_location=DEVICE)
    model.load_state_dict(state["model"])
    print(f"\nfold {fold}: nạp checkpoint epoch {state.get('epoch')} · {len(val_loader.dataset)} ca")

    result = mc_dropout_predict(
        model, val_loader, DEVICE, n_passes=N_PASSES,
        amp=bool(CFG["train"].get("amp", True)), seed=CFG.get("seed", 1337),
    )
    out = save_member_probs(OUT_ROOT / f"fold_{fold}" / "mc_dropout.npz", result)

    members = result["member_probs"]
    mean = members.mean(axis=0)
    unc = uncertainty_decomposition(members)
    print(
        f"  macro-F1 (trung bình {N_PASSES} lượt): {macro_f1(result['labels'], mean.argmax(1)):.4f}"
        f" · epistemic TB {unc['epistemic'].mean():.4f}"
        f" · {time.time() - t0:.0f}s -> {out.name}"
    )
    assert unc["epistemic"].max() > 1e-9, (
        f"fold {fold}: epistemic = 0 khắp nơi — dropout không thực sự chạy"
    )
    del model
    torch.cuda.empty_cache()

## 3. Gói mang về

Chỉ `.npz`, rất nhẹ. Ở máy local:

```
runs/E4_cv_results/fold_N/mc_dropout.npz
python -m src.eval.trust --run-dir runs/E4_cv_results --members
```

In [ ]:
import shutil

PACK = Path("/kaggle/working/mc_dropout_results")
shutil.rmtree(PACK, ignore_errors=True)
PACK.mkdir(parents=True)

for d in sorted(OUT_ROOT.glob("fold*")):
    src = d / "mc_dropout.npz"
    if src.exists():
        (PACK / d.name).mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, PACK / d.name / src.name)

total = sum(f.stat().st_size for f in PACK.rglob("*") if f.is_file())
print(f"đã gói {PACK}: {total / 2**20:.2f} MiB")
for f in sorted(PACK.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(PACK)}  {f.stat().st_size / 2**10:.0f} KiB")

print("""
⚠ TẢI VỀ: giải nén CHỈ MỘT LỚP. File .npz bản thân là zip; trình giải nén bung đệ quy
  sẽ biến nó thành thư mục và `src.eval.trust` sẽ không thấy (đã dính hai lần, S-078).
""")